# SUBLEQ Transformer: A Neural Network That Learned to Be a Computer

A **4.9M-parameter transformer** trained from scratch to execute [SUBLEQ](https://en.wikipedia.org/wiki/One-instruction_set_computer#Subtract_and_branch_if_less_than_or_equal_to_zero) — a one-instruction, Turing-complete ISA. It generalizes to run programs (Fibonacci, multiplication, division, square root) **never seen during training**.

Each instruction `(a, b, c)` does:
```
mem[b] -= mem[a]
if mem[b] <= 0: goto c
else: goto next instruction
```

One forward pass = one SUBLEQ step. Loop until halt = a general-purpose computer.

**[GitHub repo](https://github.com/anadim/subleq-transformer)**

## Setup
Clone the repo and load the trained model. Takes ~30 seconds.

In [ ]:
import os
if not os.path.exists('subleq-transformer'):
    !git clone https://github.com/anadim/subleq-transformer.git
os.chdir('subleq-transformer/round2_trained')
print('Done!')

In [ ]:
import sys
import torch
import random
from IPython.display import HTML, display

from subleq import (
    MiniSUBLEQTransformer, step as subleq_step, run as subleq_run,
    MEM_SIZE, VALUE_MIN, VALUE_MAX, CODE_SIZE, DATA_START,
    encode, decode, SEQ_LEN, VOCAB_SIZE,
    make_negate, make_addition, make_countdown, make_multiply,
    make_fibonacci, make_div, make_isqrt, make_halt,
    generate_random_program, generate_random_state,
)

# Load model
device = 'cuda' if torch.cuda.is_available() else 'cpu'
ckpt = torch.load('checkpoints/best_model.pt', map_location=device, weights_only=False)
config = ckpt.get('config', {})
model = MiniSUBLEQTransformer(
    d_model=config.get('d_model', 256),
    n_heads=config.get('n_heads', 8),
    n_layers=config.get('n_layers', 6),
    d_ff=config.get('d_ff', 1024),
    vocab_size=VOCAB_SIZE, seq_len=SEQ_LEN, dropout=0.0,
)
model.load_state_dict(ckpt['model_state'])
model.to(device)
model.eval()
print(f'Model loaded: {model.count_params():,} parameters on {device}')
print(f'Architecture: d_model={config.get("d_model",256)}, layers={config.get("n_layers",6)}, heads={config.get("n_heads",8)}')

## Helper Functions
Utilities for running the transformer and displaying results.

In [ ]:
def model_step(memory, pc):
    """Run one SUBLEQ step through the transformer."""
    inp = encode(memory, pc).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(inp)
    pred_tokens = logits.argmax(dim=-1).squeeze(0)
    return decode(pred_tokens)


def run_program(mem, pc, name='Program', max_steps=200, show_steps=True):
    """Run a program to completion, comparing model vs interpreter at each step."""
    correct = 0
    total = 0
    steps_log = []

    for s in range(max_steps):
        if pc < 0 or pc + 2 >= MEM_SIZE:
            break
        a, b = mem[pc], mem[pc + 1]
        if a < 0 or a >= MEM_SIZE or b < 0 or b >= MEM_SIZE:
            break

        # Model prediction
        model_mem, model_pc = model_step(mem, pc)
        # Ground truth
        true_mem, true_pc, halted = subleq_step(mem, pc)

        total += 1
        match = (model_mem == true_mem and model_pc == true_pc)
        if match:
            correct += 1
        steps_log.append((s + 1, pc, model_pc, true_pc, match))

        # Advance using model's predictions (not ground truth!)
        mem = model_mem
        pc = model_pc

        if halted:
            break

    # Display results
    status = '&#9989;' if correct == total else '&#10060;'
    color = 'green' if correct == total else 'red'
    html = f'<b>{name}</b>: {status} <span style="color:{color}">{correct}/{total} steps correct</span>'
    if show_steps and total <= 80:
        step_icons = ''.join(['&#9632;' if m else '<span style="color:red">&#9632;</span>' for _, _, _, _, m in steps_log])
        html += f'<br><span style="font-size:10px; letter-spacing:1px">{step_icons}</span>'
    display(HTML(html))

    return correct, total, mem, pc


def show_memory(mem, pc, title='Memory'):
    """Display memory as a colored HTML grid."""
    html = f'<div style="font-family:monospace; font-size:13px; margin:10px 0">'
    html += f'<b>{title}</b> (pc={pc})<br>'
    html += '<table style="border-collapse:collapse; margin-top:5px">'
    # Header
    html += '<tr><td></td>'
    for col in range(8):
        html += f'<td style="padding:2px 8px; color:#888; font-size:11px; text-align:center">[{col}]</td>'
    html += '</tr>'
    for row in range(4):
        row_start = row * 8
        region = 'code' if row_start < CODE_SIZE else 'data'
        rcolor = '#888' if region == 'code' else '#16a34a'
        html += f'<tr><td style="padding:2px 6px; color:{rcolor}; font-size:11px">{row_start:2d} {region}</td>'
        for col in range(8):
            idx = row_start + col
            val = mem[idx]
            bg = '#f0f9ff' if idx < CODE_SIZE else '#f0fdf4'
            if idx < CODE_SIZE and pc >= 0 and pc <= idx < pc + 3:
                bg = '#dbeafe'; style = 'font-weight:bold; color:#2563eb'
            elif idx >= DATA_START:
                style = 'color:#16a34a'
            else:
                style = 'color:#666'
            html += f'<td style="padding:3px 8px; text-align:right; background:{bg}; {style}; border:1px solid #e5e7eb">{val}</td>'
        html += '</tr>'
    html += '</table></div>'
    display(HTML(html))

---
## Demo 1: Multiplication
The transformer computes multiplication via repeated addition. It has never seen a full multiply program executed during training (only individual single-step transitions from multiply traces).

In [ ]:
a, b = 7, 9
mem, pc, result_addr = make_multiply(a, b)
show_memory(mem, pc, f'multiply({a}, {b}) — initial state')

correct, total, final_mem, final_pc = run_program(mem, pc, f'multiply({a}, {b})')
show_memory(final_mem, final_pc, f'multiply({a}, {b}) — final state')
print(f'\nResult: mem[{result_addr}] = {final_mem[result_addr]}  (expected {a * b})')

## Demo 2: Fibonacci (never in training data!)
The model has **never** seen any Fibonacci program during training — not even single steps. This is pure generalization from learning the SUBLEQ instruction.

In [ ]:
for n in range(1, 7):
    mem, pc, ra, rb = make_fibonacci(n)
    correct, total, final_mem, final_pc = run_program(mem, pc, f'fibonacci(n={n})', show_steps=True)
    f2n = final_mem[ra]
    f2n1 = final_mem[rb]
    print(f'  F({2*n}) = {f2n},  F({2*n+1}) = {f2n1}')

## Demo 3: Integer Square Root (never in training data!)
Computes floor(sqrt(n)) using the identity 1 + 3 + 5 + ... + (2k-1) = k². Never in training. Known failure: isqrt(120) returns 11 instead of 10.

In [ ]:
import math
for n in [0, 1, 2, 3, 4, 5, 8, 9, 10, 15, 16, 20, 25, 30, 36, 49, 64, 81, 100, 120]:
    mem, pc, result_addr = make_isqrt(n)
    correct, total, final_mem, final_pc = run_program(mem, pc, f'isqrt({n})')
    got = final_mem[result_addr]
    expected = math.isqrt(n)
    status = '' if got == expected else f'  *** WRONG (expected {expected}) ***'
    print(f'  isqrt({n}) = {got}{status}')

## Demo 4: Integer Division (never in training data!)
Computes a ÷ b via repeated subtraction. Known failure: div(126, 7) — the model gets stuck in a loop.

In [ ]:
div_cases = [(10, 2), (10, 3), (15, 5), (20, 7), (100, 10), (99, 9),
             (50, 7), (63, 9), (48, 6), (81, 9), (72, 8), (55, 5),
             (36, 4), (120, 11), (126, 7), (100, 13)]
for a, b in div_cases:
    mem, pc, result_addr = make_div(a, b)
    correct, total, final_mem, final_pc = run_program(mem, pc, f'div({a}, {b})')
    expected = a // b
    got = final_mem[result_addr]
    status = '' if got == expected else f'  *** WRONG (expected {expected}) ***'
    print(f'  {a} ÷ {b} = {got}{status}')

## Demo 5: Simple Programs
Negate, addition, and countdown — these appear in training data as traces.

In [ ]:
# Negate
for v in [-42, 0, 100, -128, 127]:
    mem, pc, r = make_negate(v)
    _, _, final_mem, _ = run_program(mem, pc, f'negate({v})', show_steps=False)

print()
# Addition
for a, b in [(30, 50), (-10, 10), (0, 0), (60, 60), (-60, -60)]:
    mem, pc, r = make_addition(a, b)
    _, _, final_mem, _ = run_program(mem, pc, f'add({a}, {b})', show_steps=False)

print()
# Countdown
for n in [5, 10, 20]:
    mem, pc, r = make_countdown(n)
    _, _, final_mem, _ = run_program(mem, pc, f'countdown({n})', show_steps=False)

---
## Full Test Suite
Run all the tests from the paper to verify the model.

In [ ]:
results = {}

# ─── Single-step accuracy ───
print('=== Single-Step Accuracy ===')
single_correct = 0
single_total = 0
changed_correct = 0
changed_total = 0

for n_instr in range(1, 9):
    for _ in range(500):
        mem, pc = generate_random_state(n_instr)
        true_mem, true_pc, halted = subleq_step(mem, pc)
        if halted:
            continue
        model_mem, model_pc = model_step(mem, pc)
        single_total += 1
        full_match = (model_mem == true_mem and model_pc == true_pc)
        if full_match:
            single_correct += 1
        # Check changed positions
        for i in range(MEM_SIZE):
            if true_mem[i] != mem[i]:
                changed_total += 1
                if model_mem[i] == true_mem[i]:
                    changed_correct += 1
        if true_pc != pc:
            changed_total += 1
            if model_pc == true_pc:
                changed_correct += 1

pct = 100 * single_correct / single_total if single_total > 0 else 0
cpct = 100 * changed_correct / changed_total if changed_total > 0 else 0
print(f'Full-state accuracy: {single_correct}/{single_total} ({pct:.1f}%)')
print(f'Changed-position accuracy: {changed_correct}/{changed_total} ({cpct:.1f}%)')
results['single_step'] = (single_correct, single_total)

In [ ]:
# ─── Negate ───
print('\n=== Negate ===')
nc = nt = 0
for v in range(-100, 101):
    mem, pc, r = make_negate(v)
    c, t, fm, fp = run_program(mem, pc, f'negate({v})', show_steps=False)
    # Verify answer
    true_mem, _, _ = subleq_run(list(mem), pc)
    if fm[r] == true_mem[r]:
        nc += 1
    nt += 1
print(f'\nNegate: {nc}/{nt} ({100*nc/nt:.1f}%)')
results['negate'] = (nc, nt)

In [ ]:
# ─── Addition ───
print('\n=== Addition ===')
nc = nt = 0
test_vals = list(range(-10, 11))
# Also add some edge cases
extra = [(-60, -60), (-60, 60), (60, -60), (60, 60), (0, 0)]
pairs = [(a, b) for a in test_vals for b in test_vals if abs(a + b) <= VALUE_MAX]
random.seed(42)
if len(pairs) > 300:
    pairs = random.sample(pairs, 300)
for a, b in pairs:
    mem, pc, r = make_addition(a, b)
    c, t, fm, fp = run_program(mem, pc, f'add({a},{b})', show_steps=False)
    true_mem, _, _ = subleq_run(list(mem), pc)
    if fm[r] == true_mem[r]:
        nc += 1
    nt += 1
print(f'\nAddition: {nc}/{nt} ({100*nc/nt:.1f}%)')
results['addition'] = (nc, nt)

In [ ]:
# ─── Countdown ───
print('\n=== Countdown ===')
nc = nt = 0
for n in range(1, 21):
    mem, pc, r = make_countdown(n)
    c, t, fm, fp = run_program(mem, pc, f'countdown({n})')
    true_mem, _, _ = subleq_run(list(mem), pc)
    if fm[r] == true_mem[r]:
        nc += 1
    nt += 1
print(f'\nCountdown: {nc}/{nt} ({100*nc/nt:.1f}%)')
results['countdown'] = (nc, nt)

In [ ]:
# ─── Multiply (full 12x12 table) ───
print('\n=== Multiply ===')
nc = nt = 0
for a in range(1, 13):
    for b in range(1, 13):
        if a * b > VALUE_MAX:
            continue
        mem, pc, r = make_multiply(a, b)
        c, t, fm, fp = run_program(mem, pc, f'mul({a},{b})', show_steps=False)
        true_mem, _, _ = subleq_run(list(mem), pc)
        if fm[r] == true_mem[r]:
            nc += 1
        nt += 1
print(f'\nMultiply: {nc}/{nt} ({100*nc/nt:.1f}%)')
results['multiply'] = (nc, nt)

In [ ]:
# ─── Fibonacci (NEVER in training) ───
print('\n=== Fibonacci ===')
nc = nt = 0
for n in range(1, 7):
    mem, pc, ra, rb = make_fibonacci(n)
    c, t, fm, fp = run_program(mem, pc, f'fib(n={n})')
    true_mem, _, _ = subleq_run(list(mem), pc)
    if fm[ra] == true_mem[ra] and fm[rb] == true_mem[rb]:
        nc += 1
    nt += 1
print(f'\nFibonacci: {nc}/{nt} ({100*nc/nt:.1f}%)')
results['fibonacci'] = (nc, nt)

In [ ]:
# ─── Division (NEVER in training) ───
print('\n=== Division ===')
nc = nt = 0
div_cases = [(10, 2), (10, 3), (15, 5), (20, 7), (100, 10), (99, 9),
             (50, 7), (63, 9), (48, 6), (81, 9), (72, 8), (55, 5),
             (36, 4), (120, 11), (126, 7), (100, 13)]
for a, b in div_cases:
    mem, pc, r = make_div(a, b)
    c, t, fm, fp = run_program(mem, pc, f'div({a},{b})')
    true_mem, _, _ = subleq_run(list(mem), pc)
    got = fm[r]
    expected = true_mem[r]
    if got == expected:
        nc += 1
    else:
        print(f'  FAIL: div({a},{b}) = {got}, expected {expected}')
    nt += 1
print(f'\nDivision: {nc}/{nt} ({100*nc/nt:.1f}%)')
results['division'] = (nc, nt)

In [ ]:
# ─── Square Root (NEVER in training) ───
print('\n=== Square Root ===')
import math
nc = nt = 0
for n in [0, 1, 2, 3, 4, 5, 8, 9, 10, 15, 16, 20, 25, 30, 36, 49, 64, 81, 100, 120]:
    mem, pc, r = make_isqrt(n)
    c, t, fm, fp = run_program(mem, pc, f'isqrt({n})')
    true_mem, _, _ = subleq_run(list(mem), pc)
    got = fm[r]
    expected = true_mem[r]
    if got == expected:
        nc += 1
    else:
        print(f'  FAIL: isqrt({n}) = {got}, expected {expected}')
    nt += 1
print(f'\nSquare Root: {nc}/{nt} ({100*nc/nt:.1f}%)')
results['sqrt'] = (nc, nt)

In [ ]:
# ─── Random multi-step programs ───
print('\n=== Random Multi-Step Programs ===')
random.seed(123)
nc = nt = 0
for i in range(100):
    n_instr = random.randint(1, 5)
    mem, pc = generate_random_program(n_instr)
    # Run ground truth to check if it halts
    true_mem, true_pc, true_steps = subleq_run(list(mem), pc, max_steps=30)
    if true_steps == 0:
        continue

    # Run model
    m, p = list(mem), pc
    all_match = True
    for s in range(true_steps):
        if p < 0 or p + 2 >= MEM_SIZE:
            break
        av, bv = m[p], m[p + 1]
        if av < 0 or av >= MEM_SIZE or bv < 0 or bv >= MEM_SIZE:
            break
        model_mem, model_pc = model_step(m, p)
        true_m, true_p, _ = subleq_step(m, p)
        if model_mem != true_m or model_pc != true_p:
            all_match = False
            break
        m = model_mem
        p = model_pc
    if all_match:
        nc += 1
    nt += 1

print(f'Random programs: {nc}/{nt} ({100*nc/nt:.1f}%)')
results['random'] = (nc, nt)

In [ ]:
# ─── Summary ───
print('\n' + '=' * 55)
print('                  RESULTS SUMMARY')
print('=' * 55)
total_correct = 0
total_cases = 0
for name, (c, t) in results.items():
    if name == 'single_step':
        print(f'  Single-step:     {c:5d}/{t:5d}  ({100*c/t:6.1f}%)')
    else:
        print(f'  {name:16s}: {c:5d}/{t:5d}  ({100*c/t:6.1f}%)')
        total_correct += c
        total_cases += t

print('-' * 55)
ss_c, ss_t = results.get('single_step', (0, 0))
print(f'  Single-step total: {ss_c}/{ss_t} ({100*ss_c/ss_t:.1f}%)')
print(f'  Multi-step total:  {total_correct}/{total_cases} ({100*total_correct/total_cases:.1f}%)')
print('=' * 55)

---
## Try Your Own!
Edit the cell below to run any program you want.

In [ ]:
# === TRY YOUR OWN PROGRAMS ===
# Uncomment one of these or write your own:

# Multiplication
mem, pc, r = make_multiply(11, 11)
print(f'Computing 11 × 11...')

# # Fibonacci
# mem, pc, ra, rb = make_fibonacci(6)
# r = ra  # F(12) = 144... wait, that overflows 8-bit!

# # Square root
# mem, pc, r = make_isqrt(49)

# # Division
# mem, pc, r = make_div(99, 11)

# # Random program
# mem, pc = generate_random_program(4)
# r = None

show_memory(mem, pc, 'Initial state')
correct, total, final_mem, final_pc = run_program(mem, pc, 'Your program')
show_memory(final_mem, final_pc, 'Final state')
if r is not None:
    print(f'\nResult in mem[{r}] = {final_mem[r]}')